# Macro Regime Classifier — Empirical Validation

**PRD:** PRD-050 / CC-1+2+3
**Objective:** Validate that HMM-discovered regimes align with known macro episodes (GFC 2008, COVID 2020, hiking cycle 2022-2023) WITHOUT manual labeling.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from macro_context_reader.regime.indicators import build_regime_features
from macro_context_reader.regime.hmm_classifier import HMMRegimeClassifier
from macro_context_reader.regime.analog_detector import MahalanobisAnalogDetector
from macro_context_reader.regime.consensus import (
    classify_regime_consensus,
    get_regime_history,
)

load_dotenv()

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

print('Building feature matrix from FRED (2000-present)...')
features = build_regime_features(start='2000-01-01')
print(f'Feature matrix: {features.shape[0]} months x {features.shape[1]} features')
print(f'Date range: {features.index[0].date()} -> {features.index[-1].date()}')
print(f'\nFeature stats (standardized):\n{features.describe().round(2)}')

In [ ]:
# Fit HMM with BIC selection over {3, 4, 5, 6} states
hmm = HMMRegimeClassifier()
diag = hmm.fit(features, candidate_states=(3, 4, 5, 6))

print(f'Selected n_states = {diag.n_states_selected}')
print(f'BIC scores: {diag.bic_scores}')
print(f'Converged: {diag.converged} (iters={diag.n_iter_used})')
print(f'Log-likelihood: {diag.log_likelihood:.1f}')
print(f'\nState Profiles:')
for p in diag.state_profiles:
    print(f'  State {p.state_id}: {p.label} ({p.frequency_pct:.1f}%, median duration {p.median_duration_months:.0f}mo)')
    top3 = sorted(p.mean_features.items(), key=lambda x: abs(x[1]), reverse=True)[:3]
    print(f'    Top features: {[(k, round(v, 2)) for k, v in top3]}')

# Stability check: 10 runs with different seeds
print('\nStability check (10 random seeds)...')
n_states_per_seed = []
for seed in range(10):
    h = HMMRegimeClassifier()
    d = h.fit(features, candidate_states=(3, 4, 5, 6), random_state=seed)
    n_states_per_seed.append(d.n_states_selected)
print(f'n_states across 10 seeds: {n_states_per_seed}')
print(f'Mode: {max(set(n_states_per_seed), key=n_states_per_seed.count)}')

In [ ]:
# Plot: standardized features + HMM state background coloring
history = get_regime_history(features, hmm)
states = history['hmm_state'].values
unique_states = sorted(set(states))
colors = plt.cm.Set2(np.linspace(0, 1, max(unique_states) + 1))

fig, axes = plt.subplots(features.shape[1], 1, figsize=(14, 3 * features.shape[1]), sharex=True)

for i, col in enumerate(features.columns):
    ax = axes[i]
    ax.plot(features.index, features[col], color='black', linewidth=0.8)
    ax.set_ylabel(col, fontsize=9)
    ax.grid(True, alpha=0.3)
    # Color background by regime
    for s in unique_states:
        mask = states == s
        label = hmm.get_label(s)
        for j in range(len(mask)):
            if mask[j]:
                ax.axvspan(
                    features.index[j] - pd.Timedelta(days=15),
                    features.index[j] + pd.Timedelta(days=15),
                    alpha=0.15, color=colors[s],
                    label=label if i == 0 and j == np.where(mask)[0][0] else None,
                )

axes[0].legend(loc='upper right', fontsize=8)
plt.suptitle('Standardized Features with HMM Regime Coloring', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'regime_features_hmm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Empirical validation: check known macro episodes
episodes = {
    'GFC 2008-Q4': ('2008-10-01', '2008-12-31'),
    'COVID 2020-Q2': ('2020-04-01', '2020-06-30'),
    'Hiking cycle 2022-2023': ('2022-06-01', '2023-06-30'),
    'Easing start 2024-Q4': ('2024-10-01', '2024-12-31'),
}

print('Empirical Validation — Known Macro Episodes')
print('=' * 70)

for name, (start, end) in episodes.items():
    mask = (history['date'] >= start) & (history['date'] <= end)
    episode_data = history[mask]
    if episode_data.empty:
        print(f'  {name}: NO DATA in range {start} to {end}')
        continue
    dominant = episode_data['hmm_label'].mode().iloc[0]
    dominant_pct = (episode_data['hmm_label'] == dominant).mean() * 100
    avg_prob = episode_data['max_prob'].mean()
    print(f'  {name}:')
    print(f'    Dominant regime: {dominant} ({dominant_pct:.0f}% of months)')
    print(f'    Average max_prob: {avg_prob:.3f}')
    print(f'    Labels: {list(episode_data["hmm_label"])}')

# Key check: GFC and COVID should be same state (stress-type)
gfc_state = history[(history['date'] >= '2008-10-01') & (history['date'] <= '2008-12-31')]['hmm_state'].mode()
covid_state = history[(history['date'] >= '2020-04-01') & (history['date'] <= '2020-06-30')]['hmm_state'].mode()
if not gfc_state.empty and not covid_state.empty:
    same = gfc_state.iloc[0] == covid_state.iloc[0]
    print(f'\nGFC state = {gfc_state.iloc[0]}, COVID state = {covid_state.iloc[0]}')
    print(f'Same state? {"YES" if same else "NO — investigate feature selection"}')

In [ ]:
# Analog detection for current date
from macro_context_reader.market_pricing.fx import fetch_fx_eurusd
from datetime import datetime

detector = MahalanobisAnalogDetector()
detector.fit(features)
print(f'Covariance regularized: {detector.regularized}')

# Fetch EUR/USD for forward returns
fx = fetch_fx_eurusd(datetime(2000, 1, 1))
eurusd = fx.set_index('date')['eurusd']

query_date = pd.Timestamp(features.index[-1])
print(f'\nQuery date: {query_date.date()}')

analogs = detector.find_analogs(
    query_date=query_date,
    features=features,
    k=5,
    eurusd=eurusd,
)

print(f'\nTop 5 Historical Analogs:')
print(f'{"Rank":<6}{"Date":<14}{"Distance":<12}{"EUR/USD fwd 90d":<18}')
print('-' * 50)
for a in analogs:
    fwd = f'{a.eurusd_forward_90d_pct:+.2f}%' if a.eurusd_forward_90d_pct is not None else 'N/A'
    print(f'{a.rank:<6}{str(pd.Timestamp(a.date).date()):<14}{a.distance:<12.4f}{fwd:<18}')

# Anti-regimes
anti = detector.find_anti_regimes(
    query_date=query_date,
    features=features,
    k=5,
)
print(f'\nTop 5 Anti-Regimes (most DIFFERENT from now):')
for a in anti:
    print(f'  Rank {a.rank}: {pd.Timestamp(a.date).date()} (distance={a.distance:.4f})')

## Concluzie empirica

_To be completed by Fabian after reviewing the outputs above._

**Checklist:**
- [ ] HMM labels GFC and COVID with same stress-type state
- [ ] 2022-2023 labeled as inflation-type state
- [ ] At least 3 distinct labels across all states
- [ ] Top-5 analogs are economically plausible
- [ ] BIC model selection is stable across random seeds